# 01 · Define & Explore — confidence metrics + the prediction wrapper

**Standard slot:** *define & explore.* **For Project 01 this means:** pin down what each confidence
metric means, then stand up the unified `predict()` wrapper and run it on **one** sequence as your
"hello-world" (D0).

Run `00_setup.ipynb` first in this session.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence | thermostability / ΔG |
| PAE | Å | expected error in residue-pair *relative* position | per-residue quality |
| pTM / ipTM | 0–1 | global / interface fold confidence | correctness guarantee |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

Write your own one-paragraph definitions in `D0` — including the "does not mean" column, which is
where most published mistakes live.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## The unified prediction wrapper

`scripts/predict.py` exposes `predict(sequence, tool)` → a uniform record. The real backends
(ESMFold/AF2/Boltz) need GPU + heavy installs (see `00_setup` helpers); a **mock** backend lets you
build and test the plumbing first. **Never report mock numbers as real.**

In [ ]:
from predict import predict, predict_all, Prediction

UBIQUITIN = ("MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG")

# Start with the mock backend to confirm the plumbing, THEN switch tool="esmfold".
p = predict(UBIQUITIN, tool="mock")
print(p.as_row())

### Switch to a real predictor

Once `00_setup`'s `install_esmfold()` has run in this session, change `tool="mock"` to
`tool="esmfold"` and rerun. ESMFold is the fastest real backend (no MSA), so it's the right
hello-world. Record runtime + version in `LOG.md`.

In [ ]:
# Uncomment after install_esmfold() in 00_setup:
# from predict import predict
# p = predict(UBIQUITIN, tool="esmfold")
# print(p.as_row())
# assert p.ok, p.error
print("Ready — flip to tool='esmfold' when ESMFold is installed this session.")

## Visualize a structure (py3Dmol)
Use this to eyeball any predicted PDB once you have one.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=500, height=400)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after you have a real prediction):
# show_pdb(p.pdb_path)
print("show_pdb(pdb_path) ready.")

## D0 checklist
- [ ] One-paragraph definition of each metric **with** its "does not mean" note.
- [ ] One reproduced real prediction (ESMFold) on a natural protein, with its pLDDT printed.
- [ ] `LOG.md` entry: tool version, GPU, runtime, seed.

**Next:** `02_generate.ipynb` — curate the labeled dataset and predict it all.